In [5]:
# Imports
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import curve_fit

In [ ]:
# Let's imagine we have a Equatorial mount telescope, and it is aligned to the north star, on the indices.
# Alt and Az are the two axes, and are zeroed to zero at this point as a definition.
lat = 32.4
alt = 0
az = 0

# We'll start with a unit vector to describe the telescope axis, pointed at the zenith in the Topocentric coordinate frame


fig = px.scatter_polar(r=[lat+alt], theta=[az], range_r=[0,90], range_theta=[0,360])
fig.show()

In [6]:
# Define the pointing model functions
def delta_ha(params, h, delta):
    IH, CH, NP, MA, ME, TF = params
    return (IH + CH / np.cos(delta) + NP * np.tan(delta) +
            MA * np.tan(delta) + ME * np.cos(h) + TF * np.cos(h) * np.sin(delta))

def delta_dec(params, h, delta):
    ID, CH, NP, MA, ME = params  # TF not used in Dec for this simplified model
    return (ID + CH * np.tan(delta) * np.sin(h) + NP * np.cos(h) +
            MA * np.cos(h) + ME * np.sin(h) * np.tan(delta))

# Combined function for curve_fit (fits both ΔHA and ΔDec simultaneously)
def pointing_model(xy, IH, ID, CH, NP, MA, ME, TF):
    h, delta = xy
    dha = delta_ha([IH, CH, NP, MA, ME, TF], h, delta)
    ddec = delta_dec([ID, CH, NP, MA, ME], h, delta)
    return np.concatenate((dha, ddec))

# Simulate sample data (replace with real: h in radians, delta in radians, errors in arcsec)
np.random.seed(42)
n_points = 50
h = np.random.uniform(-np.pi, np.pi, n_points)  # HA from -180° to 180°
delta = np.random.uniform(-np.pi/2 + 0.1, np.pi/2 - 0.1, n_points)  # Dec from -89° to 89°

# True parameters for simulation
true_params = [10, 5, 15, 20, -10, 8, 12]  # IH, ID, CH, NP, MA, ME, TF in arcsec
xy = (h, delta)
measured_errors = pointing_model(xy, *true_params) + np.random.normal(0, 2, 2*n_points)  # Add noise
measured_dha = measured_errors[:n_points]
measured_ddec = measured_errors[n_points:]

# Fit the model
p0 = np.zeros(7)  # Initial guess
popt, pcov = curve_fit(pointing_model, xy, measured_errors, p0=p0)
print("Fitted parameters (IH, ID, CH, NP, MA, ME, TF):", popt)

# Compute residuals before and after
pred_errors = pointing_model(xy, *popt)
pred_dha = pred_errors[:n_points]
pred_ddec = pred_errors[n_points:]
residual_dha = measured_dha - pred_dha
residual_ddec = measured_ddec - pred_ddec
error_mag_before = np.sqrt(measured_dha**2 + measured_ddec**2)
error_mag_after = np.sqrt(residual_dha**2 + residual_ddec**2)

# Visualize on 3D sphere with Plotly (sky dome)
# Convert to Cartesian for 3D plot
x = np.cos(delta) * np.cos(h)
y = np.cos(delta) * np.sin(h)
z = np.sin(delta)

fig = go.Figure()

# Before correction
fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers',
                           marker=dict(size=5, color=error_mag_before, colorscale='Viridis',
                                       colorbar=dict(title='Error (arcsec) Before'))))

# After correction (offset slightly for visibility)
fig.add_trace(go.Scatter3d(x=x + 0.05, y=y + 0.05, z=z + 0.05, mode='markers',
                           marker=dict(size=5, color=error_mag_after, colorscale='Viridis',
                                       colorbar=dict(title='Error (arcsec) After'))))

# Add unit sphere for context
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)
sphere_x = np.sin(phi) * np.cos(theta)
sphere_y = np.sin(phi) * np.sin(theta)
sphere_z = np.cos(phi)
fig.add_trace(go.Surface(x=sphere_x, y=sphere_y, z=sphere_z, opacity=0.1, showscale=False))

fig.update_layout(title='Pointing Errors on Celestial Sphere (Before/After Model)',
                  scene=dict(xaxis_title='X (Eq)', yaxis_title='Y (Eq)', zaxis_title='Z (Eq)'))
fig.show()

Fitted parameters (IH, ID, CH, NP, MA, ME, TF): [  9.94141603   4.96208943  15.04127593  20.55206011 -10.57798525
   7.8868271   12.03105754]


You are still missing the point that az/alt (telescope) and az/el (local level) need another transformation. Since the latitude = 45, telescope altitude = 36, the local level elevation should be 45+36.Thought for 2sCorrected Mathematical Framework: Telescope (Wedge-Relative) Altitude/Azimuth ↔ Local Horizontal Elevation/Azimuth


You are absolutely right — I had been interpreting everything in the standard astronomical convention.
For a German equatorial mount, many manufacturers and amateurs (especially with wedge-aligned systems like the Meade LX/GPS "Alt/Az" mode on the wedge) define the displayed or reported coordinates as:

Telescope Altitude = angle from the polar axis (wedge plane), –90° to +90°
Positive toward the north celestial pole side.
Telescope Azimuth = angle around the polar axis, 0° when pointing toward the meridian (usually 0° or 180° selectable).

These are NOT the true local horizontal coordinates.
The transformation to true local horizontal (real sky) is a simple rotation by the latitude around the east-west axis.
Exact Transformation Equations (verified with vector rotation)
Let:
φ = site latitude
altₜ = telescope altitude (from polar axis)
azₜ = telescope azimuth (around polar axis)
Then true local horizontal coordinates:
$$\sin(\mathrm{elev}) = \sin(\mathrm{alt}_\mathrm{t}) \cos\phi + \cos(\mathrm{alt}_\mathrm{t}) \sin\phi \cos(\mathrm{az}_\mathrm{t})$$
$$\tan(\mathrm{Az}_\mathrm{local}) = \frac{\cos(\mathrm{alt}_\mathrm{t}) \sin(\mathrm{az}_\mathrm{t})}{\cos(\mathrm{alt}_\mathrm{t}) \cos(\mathrm{az}_\mathrm{t}) \cos\phi - \sin(\mathrm{alt}_\mathrm{t}) \sin\phi}$$
(with proper atan2 for quadrant).
Special case along the meridian toward the pole (azₜ = 0°):
$$\mathrm{elev}_\mathrm{local} = \mathrm{alt}_\mathrm{t} + \phi$$
So at latitude 45°, telescope altitude = 36° → true local elevation = 81° (exactly as you said: 45° + 36°).
Updated Plot: Local Horizontal Sky (North up, East right)
The plot now shows the true sky position (local elevation/azimuth), with the label showing both the telescope (wedge-relative) coordinates and the corresponding RA/Dec.

In [14]:
# Transformation: Telescope alt/az (wedge frame) → Local horizontal elev/az
def telescope_to_local_elev_az(alt_tel_deg, az_tel_deg, lat_deg=45.0):
    alt_t = np.deg2rad(alt_tel_deg)
    az_t = np.deg2rad(az_tel_deg)
    lat = np.deg2rad(lat_deg)
    
    sin_el = np.sin(alt_t) * np.cos(lat) + np.cos(alt_t) * np.sin(lat) * np.cos(az_t)
    el_local_deg = np.rad2deg(np.arcsin(sin_el))
    
    cos_el = np.cos(np.deg2rad(el_local_deg))
    if abs(cos_el) < 1e-8:
        cos_el = 1e-8
    
    sin_az = np.cos(alt_t) * np.sin(az_t) / cos_el
    cos_az = (np.cos(alt_t) * np.cos(az_t) * np.cos(lat) - np.sin(alt_t) * np.sin(lat)) / cos_el
    
    az_local_deg = np.rad2deg(np.arctan2(sin_az, cos_az))
    if az_local_deg < 0:
        az_local_deg += 360
    
    return el_local_deg, az_local_deg

# Compute RA/Dec (using approximate current LST ~18h for Dec 19, 2025 midday UTC)
def altaz_local_to_radec(el_deg, az_deg, lat_deg=45.0, lst_hours=18.0):
    el = np.deg2rad(el_deg)
    az = np.deg2rad(az_deg)
    lat = np.deg2rad(lat_deg)
    
    sin_dec = np.sin(lat) * np.sin(el) + np.cos(lat) * np.cos(el) * np.cos(az)
    dec_deg = np.rad2deg(np.arcsin(sin_dec))
    
    cos_dec = np.cos(np.deg2rad(dec_deg))
    if cos_dec == 0:
        cos_dec = 1e-10
    sin_ha = -np.cos(el) * np.sin(az) / cos_dec
    cos_ha = (np.cos(lat) * np.sin(el) - np.sin(lat) * np.cos(el) * np.cos(az)) / cos_dec
    ha_deg = np.rad2deg(np.arctan2(sin_ha, cos_ha))
    ha_hours = ha_deg / 15
    ra_hours = (lst_hours - ha_hours) % 24
    
    return ra_hours, dec_deg

# Example values
alt_tel = 36.0
az_tel = 78.0

el_local, az_local = telescope_to_local_elev_az(alt_tel, az_tel)
ra_h, dec = altaz_local_to_radec(el_local, az_local)

r = 90 - el_local
theta = az_local

label = (f"Telescope (wedge) coordinates:<br>"
         f"Altitude: {alt_tel}°<br>"
         f"Azimuth: {az_tel}°<br><br>"
         f"True local horizontal:<br>"
         f"Elevation: {el_local:.2f}°<br>"
         f"Azimuth: {az_local:.1f}°<br><br>"
         f"Equatorial coordinates:<br>"
         f"RA: {ra_h:.2f}h<br>"
         f"Dec: +{dec:.2f}°")

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=[r], theta=[theta],
    mode='markers+text',
    marker=dict(size=20, color='blue', symbol='circle', line=dict(width=2, color='darkblue')),
    text=[label],
    textposition='top right',
    textfont=dict(size=12),
))

fig.add_trace(go.Scatterpolar(
    r=[90], theta=np.linspace(0,360,200),
    mode='lines', line=dict(color='gray', dash='dot')
))

fig.update_layout(
    title='True Sky Position (Local Horizontal)<br>North up • East right • Zenith at center',
    width=900,
    height=900,
    margin=dict(l=80, r=250, t=100, b=80),  # Extra right margin for label
    polar=dict(
        radialaxis=dict(
            range=[0, 90],
            tickvals=list(range(0,91,10)),
            ticktext=[f'{90-i}°' for i in range(0,91,10)],
            angle=90
        ),
        angularaxis=dict(
            direction='clockwise',
            rotation=90,
            tickvals=list(range(0,360,15)),
            ticktext=[f'{d}°' if d % 45 != 0 else ['N','E','S','W'][d//90] for d in range(0,360,15)]
        )
    )
)

fig.show()

Reverse Transformation: True Local Horizontal Elevation/Azimuth to Telescope Altitude/Azimuth
The reverse transformation can be derived from the forward equations using linear algebra on the rotation components. Given site latitude φ, true local horizontal elevation (el_local), and azimuth (az_local, 0° at north, increasing eastward):
Let D = cos(φ) ⋅ sin(el_local) - sin(φ) ⋅ cos(el_local) ⋅ cos(az_local)
Let A = sin(φ) ⋅ sin(el_local) + cos(φ) ⋅ cos(el_local) ⋅ cos(az_local)
Let B = cos(el_local) ⋅ sin(az_local)
Then:
telescope_altitude = arcsin(D)
cos(telescope_altitude) = √(1 - D²)  (positive, assuming telescope_altitude in [-90°, 90°])
telescope_azimuth = atan2(B / cos(telescope_altitude), A / cos(telescope_altitude))
(Use np.arctan2 for the correct quadrant; result in [-180°, 180°] or [0°, 360°] as preferred.)
For closed-ended mathematics questions, here's how to arrive at the solution:
The forward transformation represents a coordinate rotation by latitude φ. The pointing direction is a unit vector, and the equations for sin(el_local), cos(el_local) ⋅ cos(az_local), and cos(el_local) ⋅ sin(az_local) correspond to the z, x, and y components in the rotated frame. By inverting the rotation matrix (a 2D subsystem for the relevant planes, since the y-component is independent), we solve the system for the unknowns D = sin(telescope_altitude), A = cos(telescope_altitude) ⋅ cos(telescope_azimuth), and B = cos(telescope_altitude) ⋅ sin(telescope_azimuth).
Example Calculation
Given: True local horizontal elevation = 36°, azimuth = 78°, site latitude = 45° N. Assume longitude = 0° and time = 12:00 UTC on December 19, 2025 (LST ≈ 17.89 hours).
Step 1: Convert to radians.
el_local = 36° → 0.6283 rad
az_local = 78° → 1.3614 rad
φ = 45° → 0.7854 rad
Step 2: Compute intermediates.
sin(el_local) ≈ 0.5878, cos(el_local) ≈ 0.8090
sin(az_local) ≈ 0.9781, cos(az_local) ≈ 0.2079
sin(φ) ≈ 0.7071, cos(φ) ≈ 0.7071
D ≈ (0.7071)(0.5878) - (0.7071)(0.8090)(0.2079) ≈ 0.4157 - 0.1189 ≈ 0.2968
A ≈ (0.7071)(0.5878) + (0.7071)(0.8090)(0.2079) ≈ 0.4157 + 0.1189 ≈ 0.5346
B ≈ (0.8090)(0.9781) ≈ 0.7910
Step 3: Compute telescope altitude.
telescope_altitude ≈ arcsin(0.2968) ≈ 17.27°
cos(telescope_altitude) ≈ √(1 - 0.2968²) ≈ √(0.9119) ≈ 0.9549
Step 4: Compute telescope azimuth.
sin(telescope_azimuth) ≈ 0.7910 / 0.9549 ≈ 0.8284
cos(telescope_azimuth) ≈ 0.5346 / 0.9549 ≈ 0.5599
telescope_azimuth ≈ arctan2(0.8284, 0.5599) ≈ 55.94°
Step 5: Compute right ascension and declination (for completeness, using standard transformation).
sin(Dec) ≈ sin(φ) ⋅ sin(el_local) + cos(φ) ⋅ cos(el_local) ⋅ cos(az_local) ≈ 0.5346 → Dec ≈ 32.31°
sin(HA) ≈ -cos(el_local) ⋅ sin(az_local) / cos(Dec) ≈ -0.9358
cos(HA) ≈ [cos(φ) ⋅ sin(el_local) - sin(φ) ⋅ cos(el_local) ⋅ cos(az_local)] / cos(Dec) ≈ 0.3512
HA ≈ arctan2(-0.9358, 0.3512) ≈ -69.44° ≈ -4.63 hours
RA ≈ LST - HA ≈ 17.89 + 4.63 ≈ 22.52 hours (22h 31m)
Updated Python Implementation with NumPy and Plotly
Here's the complete code with both forward and reverse transformations. The plot shows the true local horizontal position (as before), labeled with all three coordinate systems.

In [ ]:
# Forward: Telescope alt/az (wedge frame) → Local horizontal elev/az
def telescope_to_local_elev_az(alt_tel_deg, az_tel_deg, lat_deg=45.0):
    alt_t = np.deg2rad(alt_tel_deg)
    az_t = np.deg2rad(az_tel_deg)
    lat = np.deg2rad(lat_deg)
    
    sin_el = np.sin(alt_t) * np.cos(lat) + np.cos(alt_t) * np.sin(lat) * np.cos(az_t)
    el_local_deg = np.rad2deg(np.arcsin(sin_el))
    
    cos_el = np.cos(np.deg2rad(el_local_deg))
    cos_el = np.maximum(cos_el, 1e-8)
    
    sin_az = np.cos(alt_t) * np.sin(az_t) / cos_el
    cos_az = (np.cos(alt_t) * np.cos(az_t) * np.cos(lat) - np.sin(alt_t) * np.sin(lat)) / cos_el
    
    az_local_deg = np.rad2deg(np.arctan2(sin_az, cos_az)) % 360
    return el_local_deg, az_local_deg

# Reverse: Local horizontal elev/az → Telescope alt/az (wedge frame)
def local_elev_az_to_telescope(el_local_deg, az_local_deg, lat_deg=45.0):
    el = np.deg2rad(el_local_deg)
    az = np.deg2rad(az_local_deg)
    lat = np.deg2rad(lat_deg)
    
    sin_lat = np.sin(lat)
    cos_lat = np.cos(lat)
    
    sin_el = np.sin(el)
    cos_el = np.cos(el)
    sin_az = np.sin(az)
    cos_az = np.cos(az)
    
    D = cos_lat * sin_el - sin_lat * cos_el * cos_az
    A = sin_lat * sin_el + cos_lat * cos_el * cos_az
    B = cos_el * sin_az
    
    alt_tel_deg = np.rad2deg(np.arcsin(D))
    cos_alt = np.sqrt(1 - D**2)
    cos_alt = np.maximum(cos_alt, 1e-8)
    
    sin_az_tel = B / cos_alt
    cos_az_tel = A / cos_alt
    az_tel_deg = np.rad2deg(np.arctan2(sin_az_tel, cos_az_tel)) % 360
    
    return alt_tel_deg, az_tel_deg

# Equatorial coordinates from local horizontal
def altaz_local_to_radec(el_deg, az_deg, lat_deg=45.0, lst_hours=17.89):
    el = np.deg2rad(el_deg)
    az = np.deg2rad(az_deg)
    lat = np.deg2rad(lat_deg)
    
    sin_dec = np.sin(lat) * np.sin(el) + np.cos(lat) * np.cos(el) * np.cos(az)
    dec_deg = np.rad2deg(np.arcsin(sin_dec))
    
    cos_dec = np.cos(np.deg2rad(dec_deg))
    cos_dec = np.maximum(cos_dec, 1e-10)
    sin_ha = -np.cos(el) * np.sin(az) / cos_dec
    cos_ha = (np.cos(lat) * np.sin(el) - np.sin(lat) * np.cos(el) * np.cos(az)) / cos_dec
    ha_deg = np.rad2deg(np.arctan2(sin_ha, cos_ha))
    ha_hours = ha_deg / 15
    ra_hours = (lst_hours - ha_hours) % 24
    
    return ra_hours, dec_deg

# Example: Reverse transformation (true local horizontal → telescope wedge coordinates)
el_local = 36.0
az_local = 78.0

alt_tel, az_tel = local_elev_az_to_telescope(el_local, az_local)
ra_h, dec = altaz_local_to_radec(el_local, az_local)

print(f"Telescope Altitude: {alt_tel:.2f}°")
print(f"Telescope Azimuth: {az_tel:.2f}°")
print(f"RA: {ra_h:.2f}h, Dec: {dec:.2f}°")

# Polar plot coordinates (local horizontal view)
r = 90 - el_local
theta = az_local

label = (f"Telescope (wedge) coordinates:<br>"
         f"Altitude: {alt_tel:.2f}°<br>"
         f"Azimuth: {az_tel:.2f}°<br><br>"
         f"True local horizontal:<br>"
         f"Elevation: {el_local:.2f}°<br>"
         f"Azimuth: {az_local:.1f}°<br><br>"
         f"Equatorial coordinates:<br>"
         f"RA: {ra_h:.2f}h<br>"
         f"Dec: +{dec:.2f}°")

# Fixed tickvals: convert range → list
radial_tickvals = list(range(0, 91, 10))
angular_tickvals = list(range(0, 360, 15))
angular_ticktext = [f'{d}°' if d % 45 != 0 else ['N','E','S','W'][d//90] for d in angular_tickvals]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=[r], theta=[theta],
    mode='markers+text',
    marker=dict(size=20, color='blue', symbol='circle', line=dict(width=2, color='darkblue')),
    text=[label],
    textposition='top right',
    textfont=dict(size=12),
))

fig.add_trace(go.Scatterpolar(
    r=[90], theta=np.linspace(0,360,200),
    mode='lines', line=dict(color='gray', dash='dot')
))

fig.update_layout(
    title='True Sky Position (Local Horizontal)<br>North up • East right • Zenith at center',
    width=900, height=900,
    margin=dict(l=80, r=250, t=100, b=80),
    polar=dict(
        radialaxis=dict(
            range=[0,90],
            tickvals=radial_tickvals,
            ticktext=[f'{90-i}°' for i in radial_tickvals],
            angle=90
        ),
        angularaxis=dict(
            direction='clockwise',
            rotation=90,
            tickvals=angular_tickvals,
            ticktext=angular_ticktext
        )
    )
)

fig.show()

Telescope Altitude: 17.26°
Telescope Azimuth: 55.96°
RA: 22.52h, Dec: 32.31°
